# 과제 6.2 Estimator의 이론적 배경 이해

**개요:** 이 노트북은 Estimator 프리미티브를 사용할 때 적용할 수 있는 다양한 오류 완화(error mitigation) 및 오류 억제(error suppression) 기법을 다룹니다.

*   **오류 억제 vs. 오류 완화**
*   **동적 디커플링 (Dynamical Decoupling)**
*   **파울리 트월링 (Pauli Twirling)**
*   **트월드 판독 오류 소거 (Twirled Readout Error Extinction)**
*   **제로 노이즈 외삽 (Zero Noise Extrapolation)**
*   **확률적 오류 증폭 (Probabilistic Error Amplification)**
*   **확률적 오류 상쇄 (Probabilistic Error Cancellation)**

In [ ]:
# 필요한 라이브러리 불러오기
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import EstimatorV2 as Estimator

print("라이브러리를 성공적으로 임포트했습니다.")

## 목표 1: 프리미티브 이해

프리미티브(Primitive)는 일반적인 양자 컴퓨팅 작업을 추상화하여 양자 알고리즘 개발을 단순화하는 표준화된 인터페이스입니다. Qiskit은 두 가지 주요 프리미티브를 제공합니다:

1. **Sampler**: 양자 회로의 측정 결과를 샘플링합니다
2. **Estimator**: 관측량(observable)의 기댓값을 계산합니다

프리미티브에 대한 자세한 내용은 이전 섹션 4와 5의 노트북 4.2, 5.1, 5.2에서 다루고 있습니다.

## 목표 2: 오류 완화 및 오류 억제

- **오류 완화 (Error Mitigation)**: 결과의 오류를 줄이기 위한 후처리 기법
- **오류 억제 (Error Suppression)**: 회로 실행 중 오류를 방지하기 위해 적용되는 기법

In [ ]:
# 계정이 이미 저장되어 있으면 로드합니다
service = QiskitRuntimeService()

backend = service.least_busy(operational=True, simulator=False)

### 동적 디커플링 (Dynamical Decoupling)

동적 디커플링은 유휴 상태의 qubit에 펄스 시퀀스를 삽입하여 qubit 간의 원하지 않는 상호작용을 상쇄하는 오류 억제 기법입니다. 이 기법은 일부 qubit이 놀고 있는 상태인 갭(gap)이 있는 회로에서 주로 유용합니다.

회로에 연산이 많은 경우, 동적 디커플링을 추가하면 오히려 성능이 저하될 수 있습니다.

In [ ]:
# 동적 디커플링 활성화
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.scheduling_method = "asap"
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

### 파울리 트월링 (Pauli Twirling)

트월링(Twirling)은 무작위화를 사용하여 결맞음 오류(coherent error)를 확률적 오류(stochastic error)로 변환하는 오류 억제 기법입니다. 파울리 트월링은 파울리 연산을 사용하는 트월링 방식입니다.

오류를 억제하고자 하는 게이트(일반적으로 2큐비트 게이트) 전후에 무작위 파울리 게이트를 추가하여 구현합니다.

In [ ]:
# 트월링 활성화
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100
estimator.options.twirling.strategy = "all"

### 트월드 판독 오류 소거 (Twirled Readout Error eXtinction, TREX)

TREX는 트월링과 판독 오류 보정을 결합한 측정 오류 완화 기법입니다. 측정 게이트를 파울리 X 게이트와 측정, 그리고 고전적 비트 플립 게이트로 대체하는 방식으로 작동하며, 이 시퀀스는 노이즈 없는 측정과 동등합니다.

판독 오류가 발생하면 TREX는 판독 오류 행렬을 대각화하여 역변환을 용이하게 합니다. 추가 보정 회로가 필요하므로 회로에 약간의 오버헤드가 추가됩니다.

In [ ]:
# TREX 활성화
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

### 제로 노이즈 외삽 (Zero-Noise Extrapolation, ZNE)

제로 노이즈 외삽(ZNE)은 인위적으로 노이즈 수준을 높여 회로를 실행한 후, 노이즈가 없는 결과를 추정하는 오류 완화 기법입니다. 서로 다른 노이즈 증폭 계수에서 결과를 측정한 다음, 수학적 모델을 사용하여 제로 노이즈 극한으로 외삽하여 이상적인 결과를 추정합니다.

일반적인 노이즈 증폭 방법으로는 게이트 폴딩(gate folding)과 PEA(확률적 오류 증폭)가 있습니다. 외삽에는 선형, 지수, 다항식 피팅을 사용할 수 있습니다.

In [ ]:
# ZNE 활성화
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

### 확률적 오류 증폭 (Probabilistic Error Amplification, PEA)

확률적 오류 증폭(PEA)은 제로 노이즈 외삽(ZNE)에서 사용되는 노이즈 증폭 방법으로, 게이트 폴딩보다 더 정확합니다. PEA는 세 단계로 구성됩니다:

1. 학습(Learning): `LayerNoiseLearningOptions`를 사용하여 회로 내 각 얽힘 게이트 레이어의 트월드 노이즈 모델을 학습합니다.
2. 노이즈 증폭(Noise amplification): 서로 다른 노이즈 계수에서 회로를 여러 번 실행합니다.
3. 외삽(Extrapolation): 노이즈가 있는 기댓값 결과를 제로 노이즈 극한으로 외삽하여 이상적인 결과를 추정합니다.

PEA는 ZNE에서 사용 가능한 노이즈 증폭 방법이므로, PEA를 사용하려면 ZNE를 먼저 활성화해야 합니다.

In [ ]:
# PEA 활성화
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

### 확률적 오류 상쇄 (Probabilistic Error Cancellation, PEC)

확률적 오류 상쇄(PEC)는 관측량의 기댓값을 추정할 때 사용할 수 있는 오류 완화 기법입니다. 대상 회로를 노이즈가 있는 회로들의 선형 결합으로 표현합니다. 이들은 준확률 분포(quasi-probability distribution)를 형성하며, 모든 노이즈가 있는 기댓값을 결합하여 이상적인 결과를 계산할 수 있습니다.

PEC는 샘플링 오버헤드가 회로 깊이에 따라 지수적으로 증가하므로 계산 비용이 높습니다.
`max_overhead`를 사용하여 오버헤드를 제한할 수 있습니다.

In [ ]:
# PEC 활성화
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

---
## 요약
---

이 노트북에서 다룬 내용:

## Estimator 이론 이해:

1. **오류 억제**는 회로 실행 중 오류를 방지하기 위해 적용되며, 오류 완화는 결과의 오류를 줄이기 위해 후처리에서 적용됩니다.
2. **동적 디커플링**은 노는 qubit에 펄스 시퀀스를 삽입하여 원하지 않는 상호작용을 상쇄합니다.
3. **파울리 트월링**은 무작위 파울리 게이트를 추가하여 결맞음 오류를 확률적 오류로 변환합니다.
4. **트월드 판독 오류 소거**는 트월링과 판독 오류 보정을 결합하여 판독 오류를 더 쉽게 보정할 수 있게 합니다.
5. **제로 노이즈 외삽**은 증폭된 노이즈 수준에서 회로를 실행하고 제로 노이즈로 외삽하여 노이즈 없는 결과를 추정합니다.
6. **확률적 오류 증폭**은 제로 노이즈 외삽에서 사용되는 노이즈 증폭 방법으로, 노이즈 모델 학습, 증폭, 외삽의 세 단계로 실행됩니다.
7. **확률적 오류 상쇄**는 대상 회로를 노이즈가 있는 회로들의 선형 결합으로 표현하고, 이들 노이즈 회로의 기댓값을 결합하여 이상적인 결과를 얻습니다.

---

## 연습 문제

**1) 다음 중 오류 완화가 아닌 오류 억제의 예는 무엇입니까?**

A) 제로 노이즈 외삽 (ZNE)

B) 측정 오류 완화

C) 동적 디커플링

D) 확률적 오류 상쇄

**정답:**
<details> <br/>
C) 동적 디커플링 - 실행 중에 오류를 줄이는 기법이므로


</details>

---

**2) 다음 두 회로 중 동적 디커플링의 효과가 더 큰 것은 무엇이며, 그 이유는?**

```
# 회로 A:
qc_a = QuantumCircuit(5)
qc_a.h(0)
qc_a.cx(0, 1)
# 큐비트 2, 3, 4는 유휴 상태
qc_a.measure_all()

# 회로 B:
qc_b = QuantumCircuit(5)
qc_b.h(range(5))
for i in range(4):
    qc_b.cx(i, i+1)
qc_b.barrier()
for i in range(4, 0, -1):
    qc_b.cx(i, i-1)
qc_b.measure_all()
```

A) 모든 회로가 동적 디커플링의 혜택을 받으므로 둘 다 동일하다

B) 회로 A - 펄스 시퀀스를 삽입할 수 있는 유휴 큐비트가 있으므로

C) 회로 B - 밀집된 회로일수록 더 많은 오류 억제가 필요하므로

D) 둘 다 아님 - 동적 디커플링은 시뮬레이터에서만 작동하므로

**정답:**
<details> <br/>
B) 회로 A

동적 디커플링은 유휴 큐비트가 있는 회로에서 더 유용합니다.


</details>

---

**3) PEA를 사용하기 위한 올바른 설정은 무엇입니까?**

```
A) estimator.options.resilience.pec_mitigation = True
   estimator.options.resilience.pec.amplifier = "pea"

B) estimator.options.resilience.zne_mitigation = True
   estimator.options.resilience.zne.amplifier = "pea"

C) estimator.options.zne.enable_gates = True
   estimator.options.zne.amplifier = "pea"

D) estimator.options.dynamical_decoupling.enable = True
   estimator.options.dynamical_decoupling.amplifier = "pea"
```

**정답:**
<details> <br/>
B) zne_mitigation = True 사용

PEA는 ZNE의 노이즈 증폭 방법이므로 ZNE를 먼저 활성화해야 합니다.
</details>

----

**4) 확률적 오류 상쇄(PEC)가 회로 실행 시간에 더 큰 오버헤드를 발생시키는 이유는 무엇입니까?**

A) 많은 물리적 큐비트가 필요한 양자 오류 정정 코드가 필요하므로

B) 세션 모드에서 회로를 실행해야 하므로

C) 정확한 통계를 얻기 위해 회로를 수백만 번 실행해야 하므로

D) 샘플링 오버헤드가 회로 깊이에 따라 지수적으로 증가하므로

**정답:**
<details> <br/>

D) 샘플링 오버헤드가 회로 깊이에 따라 지수적으로 증가하므로

PEC는 대상 회로를 준확률 분포를 형성하는 노이즈 회로들의 선형 결합으로 표현합니다. 오버헤드는 회로 깊이에 따라 지수적으로 증가합니다.


</details>

---